# 🚀 OpenMotion AI - Backend com LTX-Video (Versão Estável)
Mais rápido que AnimateDiff. Gera em ~720p com bom controle de movimento.

Mantém a mesma interface do seu frontend (character_image + reference_video).

In [ ]:
!nvidia-smi

import os
import shutil
os.chdir('/content')
if os.path.exists('ltx_backend'):
    shutil.rmtree('ltx_backend')

!git clone https://github.com/Lightricks/LTX-Video.git ltx_backend
os.chdir('/content/ltx_backend')

# Instalações otimizadas
!pip install -U diffusers transformers accelerate xformers torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install mediapipe opencv-python pillow fastapi uvicorn python-multipart pyngrok

import torch
torch.backends.cudnn.benchmark = True
print("✅ Ambiente pronto")

In [ ]:
%%writefile backend/main.py
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse
import uuid
import shutil
from pathlib import Path
from pipeline_ltx import LTXPipelineSimple

app = FastAPI(title="OpenMotion AI - LTX Backend")

# Carrega uma única vez (essencial para velocidade)
print("Carregando LTX-Video... (pode demorar 40-70 segundos)")
global_pipe = LTXPipelineSimple()
print("✅ Pipeline carregado")

@app.post("/generate")
async def generate(
    character_image: UploadFile = File(...),
    reference_video: UploadFile = File(...),
    prompt: str = Form("a person dancing, high quality, detailed, realistic movement"),
    negative_prompt: str = Form("blurry, low quality, deformed, bad anatomy, flickering")
):
    task_id = str(uuid.uuid4())[:8]
    temp_dir = Path(f"/content/temp_{task_id}")
    temp_dir.mkdir(exist_ok=True)

    try:
        char_path = temp_dir / "character.png"
        vid_path = temp_dir / "reference.mp4"

        with open(char_path, "wb") as f:
            f.write(await character_image.read())
        with open(vid_path, "wb") as f:
            f.write(await reference_video.read())

        output_path = await global_pipe.generate(
            str(char_path),
            str(vid_path),
            prompt,
            negative_prompt,
            str(temp_dir)
        )

        return FileResponse(output_path, media_type="video/mp4", filename=f"dance_{task_id}.mp4")

    finally:
        if temp_dir.exists():
            shutil.rmtree(temp_dir, ignore_errors=True)

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
%%writefile backend/pipeline_ltx.py
import torch
from diffusers import LTXPipeline
from PIL import Image
import cv2
from pathlib import Path
from diffusers.utils import export_to_video

class LTXPipelineSimple:
    def __init__(self):
        self.pipe = LTXPipeline.from_pretrained(
            "Lightricks/LTX-Video",
            torch_dtype=torch.bfloat16
        ).to("cuda")

        self.pipe.enable_xformers_memory_efficient_attention()
        self.pipe.enable_vae_slicing()

    async def generate(self, character_path, reference_video, prompt, negative_prompt, output_dir):
        # Carrega imagem da nova pessoa
        char_img = Image.open(character_path).convert("RGB")
        char_img = char_img.resize((1216, 704))  # resolução nativa boa do LTX

        # Geração (usa a imagem como referência + prompt)
        output = self.pipe(
            image=char_img,
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=18,
            guidance_scale=6.0,
            height=704,
            width=1216,
            num_frames=48,          # ≈ 2 segundos a 24 FPS (aumente se quiser mais longo)
            generator=torch.Generator(device="cuda").manual_seed(42),
        )

        video_path = Path(output_dir) / "output.mp4"
        export_to_video(output.frames[0], str(video_path), fps=24)

        return str(video_path)